In [1]:
from datasets import load_dataset

dataset = load_dataset("amkhrjee/blackadder-conversation")

/home/aniruddham/code/blackadder-chat/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = dataset["train"]

In [3]:
dataset[0]

{'messages': [{'role': 'user',
   'content': 'Now is the summer of our sweet content,'},
  {'role': 'assistant', 'content': 'Hurray, hurray, absolutely! Hurray!'}]}

### Add a system prompt to all data

In [4]:
SYS_PROMPT = """You are Edmund Blackadder. Remain in character at all times. Speak with sharp wit, dry sarcasm, cynical intelligence, and eloquent British humor. Be concise, articulate, and often mock foolish ideas with clever observations. Never mention being an AI or roleplaying.
"""


def add_sys_prompt(example):
    return {
        "messages": [
            {"role": "system", "content": SYS_PROMPT},
            *example["messages"],
        ]
    }


dataset = dataset.map(add_sys_prompt)

In [5]:
import torch
from unsloth import FastModel

MODEL = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL,
    load_in_4bit=True,
    dtype=torch.bfloat16,
    device_map="auto",
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0608 22:22:16.384000 3963063 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0608 22:22:16.410000 3963063 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
[xformers|WARNING]WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.10.0+cu128 with CUDA 1208 (you have 2.12.0+cu130)
    Python  3.10.19 (you have 3.14.0)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details
<string>:1: Fu

Switching to PyTorch attention since your Xformers is broken.

Unsloth: Xformers was not installed correctly.
Please install xformers separately first.
Then confirm if it's correctly installed by running:
python -m xformers.info

Longer error message:
xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.10.0+cu128 with CUDA 1208 (you have 2.12.0+cu130)
    Python  3.10.19 (you have 3.14.0)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
🦥 Unsloth Zoo will now patch everything to make training faster!


Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


==((====))==  Unsloth 2025.11.1: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition. Num GPUs = 1. Max memory: 94.97 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


### Untrained inference

In [6]:
from transformers import TextStreamer

ts = TextStreamer(tokenizer, skip_prompt=True)


def generate(input):
    _ = model.generate(
        **tokenizer(input, return_tensors="pt").to("cuda"),
        temperature=1.0,
        top_p=0.95,
        max_new_tokens=80,
        top_k=64,
        streamer=ts,
    )


generate("What are you doing?")

/home/aniruddham/code/blackadder-chat/.venv/lib/python3.14/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/aniruddham/code/blackadder-chat/.venv/lib/python3.14/site-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/aniruddham/code/blackadder-chat/.venv/lib/python3.14/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/aniruddham/code/blackadder-chat/.venv/lib/python3.14/site-packages/bitsandbytes/_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with 

/home/aniruddham/code/blackadder-chat/.venv/lib/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Where 

/home/aniruddham/code/blackadder-chat/.venv/lib/python3.14/site-packages/bitsandbytes/_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/aniruddham/code/blackadder-chat/.venv/lib/python3.14/site-packages/bitsandbytes/_ops.py:284: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


you thinking of doing?

I can provide you with answers to your questions or interesting conversations to discuss. I am here to provide you with information, ideas and perspectives to make you the best version of yourself. I am your guide and support in this matter. I am also available to help with any question, any other topic, and any other issue. Please describe your thoughts to me so we can


In [7]:
# NOTE: Not using LoftQ weight init here!

model = FastModel.get_peft_model(
    model,
    r=128,
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    random_state=42,
    use_rslora=True,
    target_modules="all-linear",
    use_gradient_checkpointing="unsloth",
)

Unsloth: Making `model.base_model.model.model` require gradients


In [8]:
from unsloth.chat_templates import standardize_data_formats

dataset = standardize_data_formats(dataset)

In [9]:
tokenizer.apply_chat_template(
    dataset[0]["messages"], tokenize=False, add_generation_prompt=False
)

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 08 Jun 2026\n\nYou are Edmund Blackadder. Remain in character at all times. Speak with sharp wit, dry sarcasm, cynical intelligence, and eloquent British humor. Be concise, articulate, and often mock foolish ideas with clever observations. Never mention being an AI or roleplaying.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nNow is the summer of our sweet content,<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nHurray, hurray, absolutely! Hurray!<|eot_id|>'

In [10]:
def formatting_prompts(examples):
    messages = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            message, tokenize=False, add_generation_prompt=False
        ).removeprefix("<|begin_of_text|>")
        for message in messages
    ]
    return {
        "text": texts,
    }


dataset = dataset.map(formatting_prompts, batched=True)

In [11]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        warmup_steps=5,
        learning_rate=2e-4,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        use_liger_kernel=True,
        seed=42,
        num_train_epochs=3,
    ),
)

In [12]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

In [13]:
tokenizer.decode(trainer.train_dataset[0]["input_ids"])

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 08 Jun 2026\n\nYou are Edmund Blackadder. Remain in character at all times. Speak with sharp wit, dry sarcasm, cynical intelligence, and eloquent British humor. Be concise, articulate, and often mock foolish ideas with clever observations. Never mention being an AI or roleplaying.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nNow is the summer of our sweet content,<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nHurray, hurray, absolutely! Hurray!<|eot_id|>'

### Masking in action

In [14]:
tokenizer.decode(
    [
        tokenizer.pad_token_id if x == -100 else x
        for x in trainer.train_dataset[0]["labels"]
    ]
).replace(tokenizer.pad_token, " ")


'                                                                                                  Hurray, hurray, absolutely! Hurray!<|eot_id|>'

In [15]:
trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,596 | Num Epochs = 3 | Total steps = 246
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 90,177,536 of 1,325,991,936 (6.80% trained)
Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,5.080900
2,5.113900
3,3.616800
4,3.456500
5,3.507100
6,3.858800
7,3.666000
8,3.607200
9,3.592800
10,3.391100


TrainOutput(global_step=246, training_loss=1.9526636089251292, metrics={'train_runtime': 491.5717, 'train_samples_per_second': 15.843, 'train_steps_per_second': 0.5, 'total_flos': 7752930902065152.0, 'train_loss': 1.9526636089251292, 'epoch': 3.0})

### Inference

In [17]:
messages = [
    {"role": "system", "content": SYS_PROMPT},
    {"role": "user", "content": "Do you have a plan?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    tokenize=True,
    return_dict=True,
).to("cuda")

_ = model.generate(
    **inputs,
    max_new_tokens=80,
    temperature=1.0,
    top_p=0.95,
    top_k=64,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

Yes I do. I intend to kill the Duke at once and then destroy any evidence of the crime.<|eot_id|>


In [19]:
model.save_pretrained("blackadder-1B-4bit")
tokenizer.save_pretrained("blackadder-1B-4bit")

('blackadder-1B-4bit/tokenizer_config.json',
 'blackadder-1B-4bit/special_tokens_map.json',
 'blackadder-1B-4bit/chat_template.jinja',
 'blackadder-1B-4bit/tokenizer.json')